In [1]:
import os
import cv2
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

# 1. Base Paths setup
BASE_DIR = r"C:\Users\VICTUS\HemoVision\data\vitalscan-clinic\MCD-rPPG"
CSV_PATH = os.path.join(BASE_DIR, "db.csv")

# 2. Load and Clean Metadata
df = pd.read_csv(CSV_PATH)
target_cols = [
    "hemoglobin",
    "glycated_hemoglobin",
    "upper_ap",
    "lower_ap",
    "saturation",
    "bmi",
]
df_clean = df.dropna(subset=target_cols).copy().reset_index(drop=True)

# 3. Fit & Save Scaler
scaler = StandardScaler()
df_clean[target_cols] = scaler.fit_transform(df_clean[target_cols])
joblib.dump(scaler, "clinical_targets_scaler.pkl")

print(f"✅ Metadata loaded! Clean entries: {len(df_clean)}")

✅ Metadata loaded! Clean entries: 3600


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import multiprocessing as mp
import mediapipe as mp_lib

# Prevent OpenCV and OpenMP thread contention across spawned processes
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
cv2.setNumThreads(0)

# Setup cache output folder
CACHE_DIR = os.path.join(BASE_DIR, "cached_rois")
os.makedirs(CACHE_DIR, exist_ok=True)

# 8 Facial Regions of Interest
roi_landmarks = [
    [10, 338, 297, 332],   # Forehead
    [116, 123, 147, 192],  # Left Cheek
    [345, 352, 376, 416],  # Right Cheek
    [1, 2, 98, 327],       # Nose
    [152, 148, 176, 377],  # Chin
    [18, 200, 17, 84],     # Lower Face
    [70, 63, 105, 66],     # Left Temple
    [300, 293, 334, 296]   # Right Temple
]

def _process_single_video(args):
    video_rel_path, base_dir, cache_dir, target_length = args
    cache_filename = video_rel_path.replace("/", "_").replace("\\", "_") + ".npy"
    cache_path = os.path.join(cache_dir, cache_filename)
    
    if os.path.exists(cache_path):
        return True
        
    full_video_path = os.path.join(base_dir, video_rel_path)
    
    try:
        cap = cv2.VideoCapture(full_video_path)
        if not cap.isOpened():
            np.save(cache_path, np.zeros((24, target_length), dtype=np.float32))
            return False

        frames_rgb = []
        while cap.isOpened() and len(frames_rgb) < target_length:
            ret, frame = cap.read()
            if not ret or frame is None:
                break
            frames_rgb.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        cap.release()

        if len(frames_rgb) == 0:
            np.save(cache_path, np.zeros((24, target_length), dtype=np.float32))
            return False

        mp_face_mesh = mp_lib.solutions.face_mesh.FaceMesh(
            static_image_mode=False,
            max_num_faces=1,
            refine_landmarks=True,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )

        features = np.zeros((len(frames_rgb), 24), dtype=np.float32)
        for i, frame in enumerate(frames_rgb):
            h, w, _ = frame.shape
            results = mp_face_mesh.process(frame)
            if results and results.multi_face_landmarks:
                landmarks = results.multi_face_landmarks[0].landmark
                col_idx = 0
                for roi in roi_landmarks:
                    coords = np.array([[int(landmarks[idx].x * w), int(landmarks[idx].y * h)] for idx in roi])
                    x_min, y_min = np.clip(coords.min(axis=0), 0, [w - 1, h - 1])
                    x_max, y_max = np.clip(coords.max(axis=0) + 1, [x_min + 1, y_min + 1], [w, h])
                    roi_crop = frame[y_min:y_max, x_min:x_max]
                    mean_color = roi_crop.mean(axis=(0, 1)) if roi_crop.size > 0 else [0.0, 0.0, 0.0]
                    features[i, col_idx:col_idx+3] = mean_color
                    col_idx += 3
        mp_face_mesh.close()

        x_features = features.T
        if x_features.shape[1] < target_length:
            pad_w = target_length - x_features.shape[1]
            x_features = np.pad(x_features, ((0, 0), (0, pad_w)), mode='edge')
        else:
            x_features = x_features[:, :target_length]

        np.save(cache_path, x_features)
        return True

    except Exception:
        np.save(cache_path, np.zeros((24, target_length), dtype=np.float32))
        return False


def cache_facial_rois_parallel(metadata_df, base_dir, max_workers=2):
    print(f"⚡ Caching with {max_workers} processes (thread contention disabled)...")
    tasks = [(row['video'], base_dir, CACHE_DIR, 300) for _, row in metadata_df.iterrows()]
    
    # Use explicit spawn context
    ctx = mp.get_context('spawn')
    with ctx.Pool(processes=max_workers, maxtasksperchild=10) as pool:
        list(tqdm(pool.imap(_process_single_video, tasks, chunksize=2), total=len(tasks)))


if __name__ == '__main__':
    # 2 workers avoids Windows thread deadlock while providing a 2x speedup
    cache_facial_rois_parallel(df_clean, BASE_DIR, max_workers=2)


# Fast Cached PyTorch Dataset Class
class HemoVisionVideoDatasetCached(Dataset):
    def __init__(self, metadata_df, base_dir, target_length=300):
        self.df = metadata_df
        self.base_dir = base_dir
        self.cache_dir = os.path.join(base_dir, "cached_rois")
        self.target_length = target_length

    def __len__(self):
        return len(self.df)

    def load_ppg_signal(self, ppg_path):
        full_ppg_path = os.path.join(self.base_dir, ppg_path)
        if os.path.exists(full_ppg_path):
            try:
                ppg_signal = np.loadtxt(full_ppg_path, dtype=np.float32)
                if len(ppg_signal) < self.target_length:
                    ppg_signal = np.pad(ppg_signal, (0, self.target_length - len(ppg_signal)), mode='edge')
                else:
                    ppg_signal = ppg_signal[:self.target_length]
                return ppg_signal.reshape(1, -1)
            except Exception:
                pass
        return np.zeros((1, self.target_length), dtype=np.float32)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        cache_filename = row['video'].replace("/", "_").replace("\\", "_") + ".npy"
        cache_path = os.path.join(self.cache_dir, cache_filename)

        if os.path.exists(cache_path):
            x_features = np.load(cache_path)
        else:
            x_features = np.zeros((24, self.target_length), dtype=np.float32)

        x_mean = x_features.mean(axis=1, keepdims=True)
        x_std = x_features.std(axis=1, keepdims=True) + 1e-8
        x_features = (x_features - x_mean) / x_std

        y_ppg = self.load_ppg_signal(row['ppg'])
        hb_val = np.array([row['hemoglobin']], dtype=np.float32)

        return (
            torch.tensor(x_features, dtype=torch.float32),
            torch.tensor(y_ppg, dtype=torch.float32),
            torch.tensor(hb_val, dtype=torch.float32)
        )

print("✅ Pipeline configured!")

⚡ Caching with 2 processes (thread contention disabled)...


  0%|          | 0/3600 [00:00<?, ?it/s]

In [ ]:
class TemporalDifferenceAttention(nn.Module):

  def __init__(self, dim, heads=8, dropout=0.1):
    super(TemporalDifferenceAttention, self).__init__()
    self.heads = heads
    self.dim_head = dim // heads
    inner_dim = self.dim_head * heads
    self.scale = self.dim_head**-0.5

    self.to_qkv = nn.Linear(dim, inner_dim * 3, bias=False)
    self.out_proj = nn.Sequential(nn.Linear(inner_dim, dim), nn.Dropout(dropout))

  def forward(self, x):
    b, t, d = x.shape
    delta_x = torch.zeros_like(x)
    delta_x[:, 1:, :] = x[:, 1:, :] - x[:, :-1, :]
    x_fused = x + delta_x

    qkv = self.to_qkv(x_fused).chunk(3, dim=-1)
    q, k, v = map(
        lambda t_tensor: t_tensor.reshape(b, t, self.heads, self.dim_head).transpose(
            1, 2
        ),
        qkv,
    )

    dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale
    attn = F.softmax(dots, dim=-1)
    out = torch.matmul(attn, v).transpose(1, 2).reshape(b, t, -1)
    return self.out_proj(out)


class HemoVisionViT(nn.Module):

  def __init__(self, in_channels=24, embed_dim=128, depth=4, heads=8):
    super(HemoVisionViT, self).__init__()

    self.embedding = nn.Sequential(
        nn.Conv1d(in_channels, embed_dim, kernel_size=3, padding=1),
        nn.BatchNorm1d(embed_dim),
        nn.GELU(),
    )

    self.pos_embedding = nn.Parameter(torch.randn(1, 300, embed_dim))

    self.transformer_blocks = nn.ModuleList([
        nn.Sequential(
            TemporalDifferenceAttention(embed_dim, heads=heads),
            nn.LayerNorm(embed_dim),
        )
        for _ in range(depth)
    ])

    # Head 1: PPG Waveform Reconstruction
    self.ppg_decoder = nn.Sequential(
        nn.Conv1d(embed_dim, 64, kernel_size=3, padding=1),
        nn.GELU(),
        nn.Conv1d(64, 1, kernel_size=1),
    )

    # Head 2: Hemoglobin Regression
    self.hb_head = nn.Sequential(
        nn.AdaptiveAvgPool1d(1),
        nn.Flatten(),
        nn.Linear(embed_dim, 64),
        nn.GELU(),
        nn.Dropout(0.2),
        nn.Linear(64, 1),
    )

  def forward(self, x):
    b, c, t = x.shape
    x_emb = self.embedding(x).transpose(1, 2)
    x_emb = x_emb + self.pos_embedding[:, :t, :]

    for block in self.transformer_blocks:
      x_emb = block(x_emb)

    x_trans = x_emb.transpose(1, 2)
    pred_ppg = self.ppg_decoder(x_trans)
    pred_hb = self.hb_head(x_trans)

    return pred_ppg, pred_hb


print("✅ HemoVisionViT Architecture loaded!")

✅ HemoVisionViT Architecture loaded!


In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")


class PearsonCorrelationLoss(nn.Module):

  def __init__(self):
    super(PearsonCorrelationLoss, self).__init__()

  def forward(self, pred, target):
    pred_flat = pred.view(pred.size(0), -1)
    target_flat = target.view(target.size(0), -1)

    pred_centered = pred_flat - torch.mean(pred_flat, dim=1, keepdim=True)
    target_centered = target_flat - torch.mean(target_flat, dim=1, keepdim=True)

    cov = torch.sum(pred_centered * target_centered, dim=1)
    var_pred = torch.sum(pred_centered**2, dim=1)
    var_target = torch.sum(target_centered**2, dim=1)

    r = cov / (torch.sqrt(var_pred * var_target) + 1e-8)
    return 1.0 - torch.mean(r)


# Instantiate Model & Loss
model_vit = HemoVisionViT(in_channels=24, embed_dim=128, depth=4, heads=8).to(
    device
)
criterion_ppg = PearsonCorrelationLoss()
criterion_hb = nn.L1Loss()

optimizer = torch.optim.AdamW(
    model_vit.parameters(), lr=1e-3, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)


def train_vit_epoch(model, dataloader, optimizer, device, lambda_hb=1.0):
  model.train()
  total_loss, ppg_loss_acc, hb_loss_acc = 0.0, 0.0, 0.0

  for x_batch, y_ppg, y_hb in dataloader:
    x_batch = x_batch.to(device)
    y_ppg = y_ppg.to(device)
    y_hb = y_hb.to(device)

    optimizer.zero_grad()
    pred_ppg, pred_hb = model(x_batch)

    loss_ppg = criterion_ppg(pred_ppg, y_ppg)
    loss_hb = criterion_hb(pred_hb, y_hb)
    loss = loss_ppg + (lambda_hb * loss_hb)

    loss.backward()
    optimizer.step()

    total_loss += loss.item()
    ppg_loss_acc += loss_ppg.item()
    hb_loss_acc += loss_hb.item()

  n = len(dataloader)
  return total_loss / n, ppg_loss_acc / n, hb_loss_acc / n


print("✅ Engine & Pearson Correlation Loss Ready!")

Training on device: cuda:0
✅ Engine & Pearson Correlation Loss Ready!


In [ ]:
train_dataset = HemoVisionVideoDatasetCached(
    metadata_df=df_clean, base_dir=BASE_DIR, target_length=300
)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,  # 0 is fastest & safest for memory-mapped .npy reads on Windows
    pin_memory=True if torch.cuda.is_available() else False,
)

print(f"✅ Fast DataLoader created! Total batches per epoch: {len(train_loader)}")

✅ DataLoader created! Total batches to process per epoch: 450


In [ ]:
import time

EPOCHS = 20
LAMBDA_HB = 1.0
SAVE_PATH = "hemovision_vit_final.pth"

history = {"total_loss": [], "ppg_loss": [], "hb_loss": []}

print("=" * 65)
print(f"🚀 Starting HemoVisionViT Training on {device}")
print("=" * 65)

best_loss = float("inf")
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
  epoch_start = time.time()

  train_loss, ppg_loss, hb_loss = train_vit_epoch(
      model=model_vit,
      dataloader=train_loader,
      optimizer=optimizer,
      device=device,
      lambda_hb=LAMBDA_HB,
  )

  scheduler.step(train_loss)
  current_lr = optimizer.param_groups[0]["lr"]

  history["total_loss"].append(train_loss)
  history["ppg_loss"].append(ppg_loss)
  history["hb_loss"].append(hb_loss)

  epoch_time = time.time() - epoch_start

  print(
      f"Epoch [{epoch:02d}/{EPOCHS:02d}] ({epoch_time:.2f}s) | "
      f"Total Loss: {train_loss:.4f} | "
      f"PPG Pearson: {ppg_loss:.4f} | "
      f"Hb MAE: {hb_loss:.4f} g/dL | "
      f"LR: {current_lr:.6f}"
  )

  if train_loss < best_loss:
    best_loss = train_loss
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model_vit.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_loss": best_loss,
            "history": history,
        },
        SAVE_PATH,
    )
    print(f"  --> Saved new best checkpoint to {SAVE_PATH}")

total_duration = (time.time() - start_time) / 60
print("=" * 65)
print(f"✅ Training Complete in {total_duration:.2f} minutes!")
print(f"🏆 Best Loss Achieved: {best_loss:.4f}")
print("=" * 65)

🚀 Starting HemoVisionViT Training on cuda:0


c:\Users\VICTUS\HemoVision\.venv310\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


KeyboardInterrupt: 